# Detección de Somnolencia — Webcam + ESP32 real (HTTP directo)

**Arquitectura:**

```
Webcam laptop → Python (MediaPipe + PyTorch) → HTTP → ESP32 real → Buzzer
```

Sin Blynk, sin Wokwi. Todo directo:
- La **webcam de la laptop** captura video
- **Python** corre el modelo y detecta 3s de ojos cerrados
- Cuando dispara, hace `requests.get('http://IP_ESP32/alarma')`
- El **ESP32** (con el sketch `funcional_con_wifi.ino`) activa el buzzer

**Requisitos previos:**
1. Subir el sketch `funcional_con_wifi.ino` al ESP32
2. Anotar la IP que aparece en el Monitor Serie
3. El ESP32 y la laptop deben estar en la misma red WiFi (hotspot del iPhone)
4. Encender el ESP32 con el botón general (LED verde encendido) antes de correr la última celda

In [ ]:
# ============ CELDA 1: Imports y configuración ============
import os
import time
import cv2
import numpy as np
import torch
import torch.nn as nn
import mediapipe as mp
import requests
from torchvision import transforms

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando: {DEVICE}")

# ---- Parámetros del modelo (deben coincidir con el entrenamiento) ----
IMG_SIZE        = 64
UMBRAL_ABIERTO  = 0.75    # confianza mínima para declarar "Abierto"
UMBRAL_SEGUNDOS = 3.0     # segundos de ojos cerrados para disparar alarma

# ---- Ruta del modelo entrenado (relativa a la carpeta detection/) ----
MODEL_PATH = "model/eye_cnn_best.pth"

CLASSES = ["Cerrado", "Abierto"]

In [2]:
# ============ CELDA 2: Definición de la arquitectura EyeCNN ============
class EyeCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 2)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

print("EyeCNN definida ✓")

EyeCNN definida ✓


In [3]:
# ============ CELDA 3: Cargar el modelo entrenado ============
model = EyeCNN().to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print(f"Modelo cargado desde {MODEL_PATH} ✓")

Modelo cargado desde eye_cnn_best.pth ✓


C:\Users\aaron\AppData\Local\Temp\ipykernel_16560\3601327155.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(MODEL_PATH, map_location=DE

## Configuración del ESP32

Antes de correr la Celda 5:

1. Sube `funcional_con_wifi.ino` al ESP32 con el Arduino IDE
2. Abre el Monitor Serie a 115200 y presiona RST
3. Copia la IP que aparece (algo como `172.20.10.X`)
4. Pégala abajo en la Celda 4

In [5]:
# ============ CELDA 4: Configurar IP del ESP32 y probar conexion ============
import requests

# Pega aqui la IP que salio en el Monitor Serie del ESP32
ESP32_IP = "192.168.139.207"   # <-- CAMBIA POR TU IP REAL

ALARMA_URL = f"http://{ESP32_IP}/alarma"
OK_URL     = f"http://{ESP32_IP}/ok"
ESTADO_URL = f"http://{ESP32_IP}/estado"

def enviar_alarma():
    try:
        r = requests.get(ALARMA_URL, timeout=1)
        return r.status_code == 200
    except Exception as e:
        print(f"Error enviando /alarma: {e}")
        return False

def enviar_ok():
    try:
        r = requests.get(OK_URL, timeout=1)
        return r.status_code == 200
    except Exception as e:
        print(f"Error enviando /ok: {e}")
        return False

def leer_estado():
    try:
        r = requests.get(ESTADO_URL, timeout=1)
        return r.text if r.status_code == 200 else None
    except Exception as e:
        print(f"Error leyendo /estado: {e}")
        return None

# Prueba de conexion
print(f"Probando conexion con ESP32 en {ESP32_IP}...")
print("Enviando /alarma...")
if enviar_alarma():
    print("  Respuesta OK, el buzzer deberia estar sonando")
else:
    print("  ERROR: no se pudo conectar. Verifica IP, WiFi y que el sistema este ENCENDIDO (LED verde)")

import time
time.sleep(2)

print("Enviando /ok...")
if enviar_ok():
    print("  Respuesta OK, buzzer apagado")

print(f"\nEstado actual: {leer_estado()}")

Probando conexion con ESP32 en 192.168.139.207...
Enviando /alarma...
  Respuesta OK, el buzzer deberia estar sonando
Enviando /ok...
  Respuesta OK, buzzer apagado

Estado actual: OFF


## Detección en tiempo real

Ejecuta la siguiente celda cuando:
- El ESP32 este encendido con el sistema ON (LED verde prendido — presiona el boton general)
- La Celda 4 haya conectado exitosamente
- Tengas el archivo `model/eye_cnn_best.pth` en la carpeta `detection/` (junto a este notebook)

In [6]:
# ============ CELDA 5: Deteccion webcam + HTTP al ESP32 ============
import time
import cv2
import numpy as np
import mediapipe as mp
from torchvision import transforms

# Landmarks de MediaPipe para ojos
LEFT_EYE  = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

UMBRAL_SEGUNDOS = 3.0
UMBRAL_ABIERTO  = 0.50   # <0.5 → cerrados; >=0.5 → abiertos

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    max_num_faces=1, refine_landmarks=True,
    min_detection_confidence=0.5, min_tracking_confidence=0.5
)

infer_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

def crop_eye(frame, landmarks, eye_idx, w, h, margin=0.6):
    xs = [landmarks[i].x * w for i in eye_idx]
    ys = [landmarks[i].y * h for i in eye_idx]
    x1, x2 = min(xs), max(xs)
    y1, y2 = min(ys), max(ys)
    mx = (x2 - x1) * margin
    my = (y2 - y1) * margin * 1.5
    x1, x2 = int(max(0, x1 - mx)), int(min(w, x2 + mx))
    y1, y2 = int(max(0, y1 - my)), int(min(h, y2 + my))
    if x2 - x1 < 10 or y2 - y1 < 10:
        return None, None
    return frame[y1:y2, x1:x2], (x1, y1, x2, y2)

model.eval()
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)   # webcam con backend Windows

if not cap.isOpened():
    print("ERROR: no se pudo abrir la webcam.")
else:
    print("Camara abierta. Presiona Q en la ventana para salir.")
    closed_since  = None
    alarma_activa = False

    while True:
        ok, frame = cap.read()
        if not ok:
            print("No se pudo leer frame")
            break

        h, w = frame.shape[:2]
        results = face_mesh.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        estado = "SIN ROSTRO"
        color = (200, 200, 200)

        if results.multi_face_landmarks:
            lms = results.multi_face_landmarks[0].landmark
            probs_abierto = []   # guardamos p_abierto de cada ojo

            for eye_idx in (LEFT_EYE, RIGHT_EYE):
                crop, box = crop_eye(frame, lms, eye_idx, w, h)
                if crop is None:
                    continue

                # Predecir probabilidad de "Abierto"
                x = infer_tf(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)).unsqueeze(0).to(DEVICE)
                with torch.no_grad():
                    prob = torch.softmax(model(x), dim=1)[0]
                p_abierto = prob[1].item()
                probs_abierto.append(p_abierto)

                # Dibujar rectangulo del ojo (siempre muestra p_abierto)
                pred_visual = 1 if p_abierto >= UMBRAL_ABIERTO else 0
                x1, y1, x2, y2 = box
                c = (0, 0, 255) if pred_visual == 0 else (0, 255, 0)
                cv2.rectangle(frame, (x1, y1), (x2, y2), c, 2)
                cv2.putText(frame, f"{CLASSES[pred_visual]} {p_abierto:.2f}",
                            (x1, y1 - 8),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, c, 1)

            # Decision GLOBAL basada en el PROMEDIO de ambos ojos
            if probs_abierto:
                p_promedio = sum(probs_abierto) / len(probs_abierto)
                ojos_cerrados = p_promedio < UMBRAL_ABIERTO

                if ojos_cerrados:
                    if closed_since is None:
                        closed_since = time.time()
                    elapsed = time.time() - closed_since
                    estado = f"OJOS CERRADOS {elapsed:.1f}s (avg {p_promedio:.2f})"
                    color = (0, 0, 255)
                    if elapsed >= UMBRAL_SEGUNDOS and not alarma_activa:
                        alarma_activa = True
                        if enviar_alarma():
                            print(">>> ALARMA ENVIADA AL ESP32")
                else:
                    closed_since = None
                    if alarma_activa:
                        alarma_activa = False
                        if enviar_ok():
                            print(">>> OK enviado (alarma apagada)")
                    estado = f"OJOS ABIERTOS (avg {p_promedio:.2f})"
                    color = (0, 255, 0)

        cv2.putText(frame, estado, (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
        cv2.imshow("Deteccion Somnolencia (HTTP) - Q para salir", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    face_mesh.close()
    enviar_ok()   # aseguramos apagar el buzzer al salir
    print("Finalizado")

Camara abierta. Presiona Q en la ventana para salir.


C:\Users\aaron\anaconda3\envs\CAPSTONE\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


>>> ALARMA ENVIADA AL ESP32
>>> OK enviado (alarma apagada)
>>> ALARMA ENVIADA AL ESP32
>>> OK enviado (alarma apagada)
>>> ALARMA ENVIADA AL ESP32
>>> OK enviado (alarma apagada)
>>> ALARMA ENVIADA AL ESP32
>>> OK enviado (alarma apagada)
>>> ALARMA ENVIADA AL ESP32
>>> OK enviado (alarma apagada)
>>> ALARMA ENVIADA AL ESP32
>>> OK enviado (alarma apagada)
>>> ALARMA ENVIADA AL ESP32
>>> OK enviado (alarma apagada)
>>> ALARMA ENVIADA AL ESP32
>>> OK enviado (alarma apagada)
>>> ALARMA ENVIADA AL ESP32
>>> OK enviado (alarma apagada)
>>> ALARMA ENVIADA AL ESP32
>>> OK enviado (alarma apagada)
>>> ALARMA ENVIADA AL ESP32
>>> OK enviado (alarma apagada)
Finalizado


## Orden de ejecución cada vez que quieras usar el sistema

1. Enciende el ESP32 y espera a que se conecte al WiFi
2. Anota la IP del Monitor Serie (si cambió)
3. **Presiona el botón general del ESP32** para encender el sistema (LED verde ON)
4. Corre Celdas 1, 2, 3 del notebook (imports, modelo, carga)
5. Corre Celda 4 (prueba conexión — deberías escuchar un beep corto)
6. Corre Celda 5 (webcam + detección)
7. Cierra los ojos 3 segundos → buzzer suena
8. Presiona Q en la ventana del video para terminar